# Data-Driven Investment Intelligence Using NIFTY-50

### Notebook Structure:
1. **Data Loading** — Load all 49 stock CSVs and metadata
2. **EDA** — Return distribution, sector analysis, volatility, correlation
3. **Feature Engineering** — 15 technical indicators per stock
4. **Stock Predictor Engine** — XGBoost regression + classification
5. **Portfolio Construction** — Conservative, Balanced, Aggressive portfolios
6. **Risk Assessment** — Sharpe, Sortino, Max Drawdown, VaR
7. **Feature Importance** — Model explainability
8. **Project Summary** — Combined dashboard

### Requirements:
- Place all stock CSVs in a `data/` folder
- Run cells in order from top to bottom

# EDA

In [ ]:
import pandas as pd
import os

DATA_DIR = "data/"

dfs = []
for fname in os.listdir(DATA_DIR):
    if fname.endswith(".csv") and fname != "stock_metadata.csv":
        symbol = fname.replace(".csv", "")
        df = pd.read_csv(os.path.join(DATA_DIR, fname), parse_dates=["Date"])
        df["Symbol"] = symbol
        dfs.append(df)

combined = pd.concat(dfs, ignore_index=True)
combined.sort_values(["Symbol", "Date"], inplace=True)
combined.reset_index(drop=True, inplace=True)

meta = pd.read_csv(os.path.join(DATA_DIR, "stock_metadata.csv"))

print("✅ Data loaded!")
print(f"Total rows: {len(combined):,}")
print(f"Stocks: {combined['Symbol'].nunique()}")
print(f"Date range: {combined['Date'].min()} → {combined['Date'].max()}")
print(f"\nColumns: {combined.columns.tolist()}")
print(f"\nMetadata:\n{meta.head()}")

In [ ]:
# Merge sector info
combined = combined.merge(meta[['Symbol', 'Industry']], on='Symbol', how='left')

# Calculate daily return
combined['Daily_Return'] = combined.groupby('Symbol')['Close'].pct_change()

print("✅ Returns calculated!")
print(combined[['Date', 'Symbol', 'Close', 'Daily_Return', 'Industry']].head(10))
print(f"\nNull returns: {combined['Daily_Return'].isnull().sum()}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Distribution of all daily returns
axes[0].hist(combined['Daily_Return'].dropna(), bins=200, color='steelblue', edgecolor='none')
axes[0].set_title('Distribution of Daily Returns — All Stocks')
axes[0].set_xlabel('Daily Return')
axes[0].set_ylabel('Frequency')
axes[0].axvline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlim(-0.2, 0.2)  # focus on -20% to +20%

# Plot 2: Average annualised return by sector
sector_ret = combined.groupby('Industry')['Daily_Return'].mean() * 252
sector_ret.sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Annualised Mean Return by Sector')
axes[1].set_xlabel('Annualised Return')
axes[1].axvline(0, color='black', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('eda_returns.png', dpi=150)
plt.show()

print("\nSector Returns:")
print(sector_ret.sort_values(ascending=False).round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot 1: Volatility by stock
vol = combined.groupby('Symbol')['Daily_Return'].std() * (252**0.5)
vol.sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Annualised Volatility by Stock')
axes[0].set_ylabel('Volatility (σ)')
axes[0].set_xlabel('Stock')
axes[0].tick_params(axis='x', rotation=90)

# Plot 2: Correlation heatmap
returns_pivot = combined.pivot_table(index='Date', columns='Symbol', values='Daily_Return')
corr = returns_pivot.corr()

sns.heatmap(corr, ax=axes[1], cmap='RdYlGn', center=0,
            linewidths=0.3, annot=False, square=True)
axes[1].set_title('Return Correlation — All NIFTY-50 Stocks')

plt.tight_layout()
plt.savefig('eda_volatility_correlation.png', dpi=150)
plt.show()

# Print most and least volatile
print("Top 5 Most Volatile:")
print(vol.sort_values(ascending=False).head())
print("\nTop 5 Least Volatile:")
print(vol.sort_values().head())

In [ ]:
# Remove NIFTY50_all — it's the index file, not a stock
combined = combined[combined['Symbol'] != 'NIFTY50_all'].copy()

print(f"Stocks remaining: {combined['Symbol'].nunique()}")
print(f"Rows remaining: {len(combined):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Volatility by stock (clean)
vol = combined.groupby('Symbol')['Daily_Return'].std() * (252**0.5)
vol.sort_values(ascending=False).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Annualised Volatility by Stock (Fixed)')
axes[0].set_ylabel('Volatility (σ)')
axes[0].tick_params(axis='x', rotation=90)

# Correlation heatmap (clean)
returns_pivot = combined.pivot_table(index='Date', columns='Symbol', values='Daily_Return')
corr = returns_pivot.corr()
sns.heatmap(corr, ax=axes[1], cmap='RdYlGn', center=0,
            linewidths=0.3, annot=False, square=True)
axes[1].set_title('Return Correlation — NIFTY-50 Stocks (Fixed)')

plt.tight_layout()
plt.savefig('eda_volatility_correlation.png', dpi=150)
plt.show()

print("Top 5 Most Volatile:")
print(vol.sort_values(ascending=False).head())
print("\nTop 5 Least Volatile:")
print(vol.sort_values().head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Plot 1: Normalised price of 5 well-known stocks
stocks_to_plot = ['RELIANCE', 'TCS', 'HDFCBANK', 'INFY', 'WIPRO']
pivot = combined.pivot_table(index='Date', columns='Symbol', values='Close')
normalised = (pivot / pivot.bfill().iloc[0]) * 100

for stock in stocks_to_plot:
    if stock in normalised.columns:
        axes[0].plot(normalised.index, normalised[stock], label=stock, linewidth=1.2)

axes[0].set_title('Normalised Price Performance — Top 5 Stocks (Base 100)')
axes[0].set_ylabel('Indexed Price')
axes[0].legend()
axes[0].axvspan(pd.Timestamp('2008-09-01'), pd.Timestamp('2009-03-01'),
                alpha=0.2, color='red', label='2008 Crisis')
axes[0].axvspan(pd.Timestamp('2020-02-01'), pd.Timestamp('2020-04-01'),
                alpha=0.2, color='orange', label='COVID Crash')

# Plot 2: Market-wide average daily return over time
market_return = combined.groupby('Date')['Daily_Return'].mean()
market_return.rolling(30).mean().plot(ax=axes[1], color='darkblue', linewidth=1)
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[1].fill_between(market_return.rolling(30).mean().index,
                     market_return.rolling(30).mean(), 0,
                     where=market_return.rolling(30).mean() >= 0,
                     color='green', alpha=0.3)
axes[1].fill_between(market_return.rolling(30).mean().index,
                     market_return.rolling(30).mean(), 0,
                     where=market_return.rolling(30).mean() < 0,
                     color='red', alpha=0.3)
axes[1].set_title('30-Day Rolling Average Market Return (Green=Bull, Red=Bear)')
axes[1].set_ylabel('Avg Daily Return')

plt.tight_layout()
plt.savefig('eda_price_history.png', dpi=150)
plt.show()

# Feature Engineering

In [ ]:
import subprocess
subprocess.run(["pip", "install", "ta"])

In [ ]:
import importlib
import ta
importlib.reload(ta)
print("✅ ta imported!")

In [ ]:
import ta

def add_features(df):
    result = []
    for symbol, group in df.groupby('Symbol'):
        g = group.copy().sort_values('Date')

        # Trend
        g['MA_20']       = g['Close'].rolling(20).mean()
        g['MA_50']       = g['Close'].rolling(50).mean()
        g['EMA_12']      = g['Close'].ewm(span=12).mean()
        g['EMA_26']      = g['Close'].ewm(span=26).mean()
        g['MACD']        = g['EMA_12'] - g['EMA_26']
        g['MACD_Signal'] = g['MACD'].ewm(span=9).mean()

        # Momentum
        g['RSI'] = ta.momentum.rsi(g['Close'], window=14)

        # Volatility
        g['BB_High']     = ta.volatility.bollinger_hband(g['Close'])
        g['BB_Low']      = ta.volatility.bollinger_lband(g['Close'])
        g['ATR']         = ta.volatility.average_true_range(g['High'], g['Low'], g['Close'])

        # Volume
        g['OBV']         = ta.volume.on_balance_volume(g['Close'], g['Volume'])

        # Lag features
        for lag in [1, 3, 5]:
            g[f'Return_Lag_{lag}'] = g['Daily_Return'].shift(lag)

        # Rolling volatility
        g['Volatility_20'] = g['Daily_Return'].rolling(20).std()

        # Target variables
        g['Next_Return']   = g['Daily_Return'].shift(-1)      # regression target
        g['Direction']     = (g['Next_Return'] > 0).astype(int)  # classification target

        result.append(g)

    return pd.concat(result)

print("Adding features... ")
featured = add_features(combined)
featured.dropna(inplace=True)
featured.reset_index(drop=True, inplace=True)

print(f"✅ Features added!")
print(f"Rows after dropping NaN: {len(featured):,}")
print(f"Total columns: {len(featured.columns)}")
print(featured.columns.tolist())

In [ ]:
import subprocess
subprocess.run(["pip", "install", "xgboost"])

In [ ]:
import importlib
import xgboost
importlib.reload(xgboost)
print("✅ xgboost ready!")

# Stock Predictor Engine (XGBoost)

In [ ]:
from xgboost import XGBRegressor, XGBClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, accuracy_score
import numpy as np

FEATURE_COLS = [
    'MA_20', 'MA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_Signal',
    'RSI', 'BB_High', 'BB_Low', 'ATR', 'OBV',
    'Return_Lag_1', 'Return_Lag_3', 'Return_Lag_5', 'Volatility_20'
]

# Fix dtypes once before the loop
featured[FEATURE_COLS] = featured[FEATURE_COLS].astype(float)

regression_results = {}
classification_results = {}

print("Training models for all 49 stocks...")
print("(This will take 2-3 minutes)\n")

for symbol in featured['Symbol'].unique():
    stock = featured[featured['Symbol'] == symbol].copy()
    if len(stock) < 200:
        continue

    X = stock[FEATURE_COLS]
    y_reg = stock['Next_Return']
    y_clf = stock['Direction']

    tscv = TimeSeriesSplit(n_splits=5)
    mae_scores, rmse_scores, acc_scores = [], [], []

    for train_idx, test_idx in tscv.split(X):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train_r, y_test_r = y_reg.iloc[train_idx], y_reg.iloc[test_idx]
        y_train_c, y_test_c = y_clf.iloc[train_idx], y_clf.iloc[test_idx]

        reg = XGBRegressor(n_estimators=100, learning_rate=0.05,
                           max_depth=4, random_state=42, verbosity=0)
        reg.fit(X_train, y_train_r)
        preds_r = reg.predict(X_test)
        mae_scores.append(mean_absolute_error(y_test_r, preds_r))
        rmse_scores.append(np.sqrt(mean_squared_error(y_test_r, preds_r)))

        clf = XGBClassifier(n_estimators=100, learning_rate=0.05,
                            max_depth=4, random_state=42, verbosity=0,
                            eval_metric='logloss')
        clf.fit(X_train, y_train_c)
        preds_c = clf.predict(X_test)
        acc_scores.append(accuracy_score(y_test_c, preds_c))

    regression_results[symbol] = {
        'MAE':  round(np.mean(mae_scores), 5),
        'RMSE': round(np.mean(rmse_scores), 5)
    }
    classification_results[symbol] = {
        'Directional_Accuracy': round(np.mean(acc_scores), 4)
    }

reg_df = pd.DataFrame(regression_results).T.sort_values('RMSE')
clf_df = pd.DataFrame(classification_results).T.sort_values('Directional_Accuracy', ascending=False)

print("✅ Training complete!\n")
print("Top 10 Best Predicted Stocks (lowest RMSE):")
print(reg_df.head(10))
print(f"\nMean RMSE across all stocks:  {reg_df['RMSE'].mean():.5f}")
print(f"Mean MAE across all stocks:   {reg_df['MAE'].mean():.5f}")
print("\nTop 10 Directional Accuracy:")
print(clf_df.head(10))
print(f"\nMean Directional Accuracy: {clf_df['Directional_Accuracy'].mean():.4f}")

reg_df.to_csv('predictor_results.csv')
clf_df.to_csv('direction_results.csv')

# Visualise Model Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RMSE by stock
reg_df['RMSE'].sort_values().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('RMSE by Stock (Lower = Better)')
axes[0].set_ylabel('RMSE')
axes[0].tick_params(axis='x', rotation=90)
axes[0].axhline(reg_df['RMSE'].mean(), color='red', linestyle='--', label=f'Mean={reg_df["RMSE"].mean():.4f}')
axes[0].legend()

# Directional Accuracy by stock
clf_df['Directional_Accuracy'].sort_values(ascending=False).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Directional Accuracy by Stock')
axes[1].set_ylabel('Accuracy')
axes[1].tick_params(axis='x', rotation=90)
axes[1].axhline(0.5, color='black', linestyle='--', label='Random Baseline (50%)')
axes[1].axhline(clf_df['Directional_Accuracy'].mean(), color='red', linestyle='--',
                label=f'Mean={clf_df["Directional_Accuracy"].mean():.4f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_results.png', dpi=150)
plt.show()

# Portfolio Construction

In [ ]:
from scipy.optimize import minimize

# Use last 5 years of data for portfolio construction
recent = combined[combined['Date'] >= '2016-01-01'].copy()

# Build returns pivot — only stocks with complete data
returns_pivot = recent.pivot_table(index='Date', columns='Symbol', values='Daily_Return')
returns_pivot = returns_pivot.dropna(axis=1)  # stocks with no missing dates

symbols_list = returns_pivot.columns.tolist()
mean_returns = returns_pivot.mean()
cov_matrix = returns_pivot.cov()
n = len(symbols_list)

print(f"Stocks used for portfolio: {n}")
print(f"Symbols: {symbols_list}")

# ── Helper ──────────────────────────────────────────────
def portfolio_metrics(weights, mean_returns, cov_matrix, rf=0.06):
    ret = np.dot(weights, mean_returns) * 252
    vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
    sharpe = (ret - rf) / vol
    return ret, vol, sharpe

constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
bounds = [(0.01, 0.20)] * n  # min 1%, max 20% per stock

# ── Portfolio 1: Conservative (Min Volatility) ──────────
def min_vol(weights):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))

cons_result = minimize(min_vol, x0=np.ones(n)/n,
                       method='SLSQP', bounds=bounds, constraints=constraints)

# ── Portfolio 2: Aggressive (Max Sharpe) ────────────────
def neg_sharpe(weights):
    ret, vol, sharpe = portfolio_metrics(weights, mean_returns, cov_matrix)
    return -sharpe

agg_result = minimize(neg_sharpe, x0=np.ones(n)/n,
                      method='SLSQP', bounds=bounds, constraints=constraints)

# ── Portfolio 3: Balanced (Equal Weight) ────────────────
bal_weights = np.ones(n) / n

# ── Summary Table ────────────────────────────────────────
portfolios = {
    'Conservative': cons_result.x,
    'Balanced':     bal_weights,
    'Aggressive':   agg_result.x
}

print(f"\n{'Portfolio':<15} {'Return':>10} {'Volatility':>12} {'Sharpe':>10}")
print("-" * 50)
for name, w in portfolios.items():
    ret, vol, sharpe = portfolio_metrics(w, mean_returns, cov_matrix)
    print(f"{name:<15} {ret:>10.2%} {vol:>12.2%} {sharpe:>10.3f}")

# ── Plot allocations ─────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['green', 'steelblue', 'coral']

for ax, (name, w), color in zip(axes, portfolios.items(), colors):
    alloc = pd.Series(w, index=symbols_list).sort_values(ascending=False)
    alloc = alloc[alloc > 0.015]  # show only meaningful allocations
    alloc.plot(kind='bar', ax=ax, color=color)
    ret, vol, sharpe = portfolio_metrics(w, mean_returns, cov_matrix)
    ax.set_title(f'{name}\nReturn: {ret:.1%} | Vol: {vol:.1%} | Sharpe: {sharpe:.2f}')
    ax.set_ylabel('Weight')
    ax.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('portfolios.png', dpi=150)
plt.show()

# Saved allocations
for name, w in portfolios.items():
    pd.Series(w, index=symbols_list).sort_values(ascending=False).to_csv(
        f'portfolio_{name.lower()}.csv', header=['Weight'])

# Risk Assessment

In [ ]:
risk_records = []

for symbol in combined['Symbol'].unique():
    stock = combined[combined['Symbol'] == symbol]['Daily_Return'].dropna()
    if len(stock) < 100:
        continue

    rf_daily = 0.06 / 252

    # Annualised volatility
    vol = stock.std() * np.sqrt(252)

    # Sharpe Ratio
    sharpe = (stock.mean() - rf_daily) / stock.std() * np.sqrt(252)

    # Sortino Ratio (downside only)
    downside = stock[stock < 0].std() * np.sqrt(252)
    sortino = (stock.mean() * 252 - 0.06) / downside

    # Maximum Drawdown
    cum = (1 + stock).cumprod()
    rolling_max = cum.cummax()
    drawdown = (cum - rolling_max) / rolling_max
    max_dd = drawdown.min()

    # Value at Risk 95%
    var_95 = np.percentile(stock, 5)

    risk_records.append({
        'Symbol':     symbol,
        'Annual_Vol': round(vol, 4),
        'Sharpe':     round(sharpe, 3),
        'Sortino':    round(sortino, 3),
        'Max_DD':     round(max_dd, 4),
        'VaR_95':     round(var_95, 4)
    })

risk_df = pd.DataFrame(risk_records).set_index('Symbol')
risk_df.to_csv('risk_metrics.csv')

print("Top 10 by Sharpe Ratio:")
print(risk_df.sort_values('Sharpe', ascending=False).head(10))
print("\nTop 10 Least Risky (lowest volatility):")
print(risk_df.sort_values('Annual_Vol').head(10))
print("\nTop 10 Worst Max Drawdown:")
print(risk_df.sort_values('Max_DD').head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Risk vs Return scatter
annual_ret = combined.groupby('Symbol')['Daily_Return'].mean() * 252
annual_vol = combined.groupby('Symbol')['Daily_Return'].std() * np.sqrt(252)

axes[0].scatter(annual_vol, annual_ret, alpha=0.7, s=80, color='steelblue')
for sym in annual_vol.index:
    axes[0].annotate(sym, (annual_vol[sym], annual_ret[sym]),
                     fontsize=6.5, alpha=0.85)
axes[0].set_xlabel('Annualised Volatility (Risk)')
axes[0].set_ylabel('Annualised Return')
axes[0].set_title('Risk vs Return — NIFTY-50 Stocks')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.4)

# Sharpe Ratio bar chart
risk_df['Sharpe'].sort_values(ascending=False).plot(
    kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Sharpe Ratio by Stock')
axes[1].set_ylabel('Sharpe Ratio')
axes[1].tick_params(axis='x', rotation=90)
axes[1].axhline(0, color='black', linestyle='--', alpha=0.5)
axes[1].axhline(risk_df['Sharpe'].mean(), color='red', linestyle='--',
                label=f'Mean={risk_df["Sharpe"].mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('risk_assessment.png', dpi=150)
plt.show()

In [ ]:
# Feature Importance — Explainability 
symbol = 'HDFCBANK'
stock = featured[featured['Symbol'] == symbol].copy()
X = stock[FEATURE_COLS].astype(float)
y = stock['Next_Return']

split = int(len(X) * 0.8)
final_model = XGBRegressor(n_estimators=100, learning_rate=0.05,
                           max_depth=4, random_state=42, verbosity=0)
final_model.fit(X.iloc[:split], y.iloc[:split])

# Plot
importance = pd.Series(final_model.feature_importances_, index=FEATURE_COLS)
importance.sort_values(ascending=True).plot(
    kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Feature Importance — HDFCBANK (XGBoost)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()

print("Top 5 most important features:")
print(importance.sort_values(ascending=False).head())

# Project Summary

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle('NIFTY-50 Investment Intelligence — Project Summary', 
             fontsize=16, fontweight='bold', y=0.98)

# 1. Sector Returns
ax1 = fig.add_subplot(2, 3, 1)
sector_ret = combined.groupby('Industry')['Daily_Return'].mean() * 252
sector_ret.sort_values().plot(kind='barh', ax=ax1, color='coral')
ax1.set_title('Annualised Return by Sector')
ax1.set_xlabel('Return')

# 2. Risk vs Return
ax2 = fig.add_subplot(2, 3, 2)
annual_ret = combined.groupby('Symbol')['Daily_Return'].mean() * 252
annual_vol = combined.groupby('Symbol')['Daily_Return'].std() * np.sqrt(252)
ax2.scatter(annual_vol, annual_ret, alpha=0.7, s=60, color='steelblue')
for sym in ['NESTLEIND', 'SHREECEM', 'BAJFINANCE', 'COALINDIA', 'WIPRO', 'HDFCBANK']:
    if sym in annual_vol.index:
        ax2.annotate(sym, (annual_vol[sym], annual_ret[sym]), fontsize=7)
ax2.set_title('Risk vs Return')
ax2.set_xlabel('Volatility')
ax2.set_ylabel('Return')
ax2.axhline(0, color='red', linestyle='--', alpha=0.4)

# 3. Portfolio Comparison
ax3 = fig.add_subplot(2, 3, 3)
port_names = ['Conservative', 'Balanced', 'Aggressive']
port_returns = []
port_vols = []
port_sharpes = []
for name, w in portfolios.items():
    ret, vol, sharpe = portfolio_metrics(w, mean_returns, cov_matrix)
    port_returns.append(ret)
    port_vols.append(vol)
    port_sharpes.append(sharpe)

x = np.arange(3)
width = 0.3
ax3.bar(x - width, port_returns, width, label='Return', color='green', alpha=0.7)
ax3.bar(x, port_vols, width, label='Volatility', color='orange', alpha=0.7)
ax3.bar(x + width, port_sharpes, width, label='Sharpe', color='steelblue', alpha=0.7)
ax3.set_xticks(x)
ax3.set_xticklabels(port_names)
ax3.set_title('Portfolio Comparison')
ax3.legend(fontsize=8)

# 4. Feature Importance
ax4 = fig.add_subplot(2, 3, 4)
importance.sort_values(ascending=True).tail(8).plot(kind='barh', ax=ax4, color='steelblue')
ax4.set_title('Top 8 Predictive Features\n(HDFCBANK)')
ax4.set_xlabel('Importance')

# 5. Sharpe Ratio Top 15
ax5 = fig.add_subplot(2, 3, 5)
risk_df['Sharpe'].sort_values(ascending=False).head(15).plot(
    kind='bar', ax=ax5, color='coral')
ax5.set_title('Top 15 Stocks by Sharpe Ratio')
ax5.set_ylabel('Sharpe')
ax5.tick_params(axis='x', rotation=90)
ax5.axhline(0, color='black', linestyle='--', alpha=0.5)

# 6. Model RMSE Top 15
ax6 = fig.add_subplot(2, 3, 6)
reg_df['RMSE'].sort_values().head(15).plot(kind='bar', ax=ax6, color='green', alpha=0.8)
ax6.set_title('Top 15 Most Predictable Stocks\n(Lowest RMSE)')
ax6.set_ylabel('RMSE')
ax6.tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.savefig('project_summary.png', dpi=150)
plt.show()
